In [ ]:
!pip install ripser scikit-learn scipy tqdm matplotlib torch torchvision -q

import copy, json, os, random, sys, time, warnings
import numpy as np
import torch, torch.nn as nn, torch.optim as optim
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from torch.utils.data import DataLoader, TensorDataset, Subset
from tqdm.notebook import tqdm
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")
if str(DEVICE) == 'cpu':
    print("⚠  No GPU detected. Scenarios C/D will be very slow. "
          "Go to Runtime → Change runtime type → T4 GPU.")

def set_seed(s=42):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
set_seed()

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.1/842.1 kB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.6/48.6 kB 4.0 MB/s eta 0:00:00
Device: cuda


In [ ]:
from scipy.spatial.distance import pdist, squareform
from scipy.sparse.csgraph import minimum_spanning_tree

class TopologicalFeatureExtractor:
    """48-dim PH descriptor. H0 via MST; H1 via Ripser."""
    def __init__(self, n_points=80, n_thresholds=20, use_ripser=True):
        self.n_points = n_points; self.n_thresholds = n_thresholds
        self.use_ripser = use_ripser
        try: import ripser; self._rip = True
        except ImportError: self._rip = False

    def fit_transform(self, X):
        X = np.asarray(X, dtype=np.float32)
        if X.ndim == 1: X = X.reshape(-1,1)
        if len(X) > self.n_points:
            X = X[np.random.choice(len(X), self.n_points, replace=False)]
        if len(X) < 4: return np.zeros(48, dtype=np.float32)
        norms = np.linalg.norm(X, axis=1, keepdims=True)
        X = X / (norms + 1e-8)
        return self._assemble(self._h0(X), self._h1(X))

    def _h0(self, X):
        mst = minimum_spanning_tree(squareform(pdist(X))).toarray()
        return np.sort(mst[mst > 0])

    def _h1(self, X):
        if self.use_ripser and self._rip:
            from ripser import ripser as rips
            dgm = rips(X, maxdim=1)['dgms'][1]
            if len(dgm) == 0: return np.array([])
            fin = dgm[~np.isinf(dgm[:,1])]
            return (fin[:,1]-fin[:,0]) if len(fin)>0 else np.array([])
        # fallback approximation
        dist = squareform(pdist(X)); pers = []
        cap = min(len(X), 25)
        for i in range(cap-2):
            for j in range(i+1,cap-1):
                for k in range(j+1,cap):
                    s = sorted([dist[i,j],dist[j,k],dist[i,k]])
                    if s[2]>s[1]: pers.append(s[2]-s[1])
        return np.array(pers)

    def _assemble(self, h0, h1):
        d = np.zeros(48, dtype=np.float32)
        for j, h in enumerate([h0, h1]):
            if len(h) == 0: continue
            d[j]   = len(h) + (1 if j==0 else 0)
            p      = h/(h.sum()+1e-10)
            d[2+j] = -np.sum(p*np.log(p+1e-10))
            d[4+j] = np.sqrt(np.sum(h**2))
            d[6+j] = np.sum(h > np.median(h))
            t = np.linspace(0, max(np.percentile(h,95),1e-6), self.n_thresholds)
            for i,tv in enumerate(t):
                d[8 + j*20 + i] = np.sum(h > tv)
        if len(h0)==0: d[0]=1
        return d

In [ ]:
from sklearn.cluster import AgglomerativeClustering

def flatten_params(sd):
    trainable_params = []
    for k, v in sd.items():
        # Heuristic to identify and exclude buffers, common in BatchNorm layers
        if 'running_mean' in k or 'running_var' in k or 'num_batches_tracked' in k:
            continue
        trainable_params.append(v.flatten().float())
    return torch.cat(trainable_params).cpu().numpy()

def unflatten_params(flat, ref_sd):
    new_sd, off = {}, 0
    for k,v in ref_sd.items():
        # Apply the same filter logic for unflattening to match
        if 'running_mean' in k or 'running_var' in k or 'num_batches_tracked' in k:
            new_sd[k] = v.clone() # Buffers are not in `flat`, so copy from reference
            continue
        n = v.numel()
        new_sd[k] = torch.tensor(flat[off:off+n],dtype=v.dtype).reshape(v.shape)
        off += n
    return new_sd

class TopoFLServer:
    def __init__(self, n_clusters=2, blend_coeff=0.3, anomaly_tau=2.0,
                 warmup_rounds=10, tda_kwargs=None):
        self.nc=n_clusters; self.bc=blend_coeff; self.tau=anomaly_tau
        self.warmup=warmup_rounds
        self.ext = TopologicalFeatureExtractor(**(tda_kwargs or {}))
        self._asgn=None; self._hist=[]

    def aggregate(self, descs, params, sizes, round_idx=0):
        K    = len(params)
        desc = np.stack(descs)
        self._hist.append(desc.copy())
        # Use the locally defined flatten_params, now corrected
        flat = np.stack([flatten_params(p) for p in params])
        ref  = params[0]

        # warm-up: plain FedAvg
        if round_idx < self.warmup:
            w   = np.array(sizes,dtype=float); w /= w.sum()
            agg = (w[:,None]*flat).sum(0)
            sd  = unflatten_params(agg, ref)
            return sd, {0:sd}, dict(mode='warmup', asgn=np.zeros(K,int))

        # clustering
        if round_idx==self.warmup or self._asgn is None:
            self._asgn = self._cluster(desc)
        else:
            if len(self._hist)>=2:
                drift = np.linalg.norm(desc-self._hist[-2],axis=1)
                dz    = (drift-drift.mean())/(drift.std()+1e-10)
                if dz.max() > self.tau: self._asgn = self._cluster(desc)
        asgn = self._asgn

        # trust scores
        delta = np.array([np.mean([np.linalg.norm(desc[k]-desc[j])
                          for j in range(K) if j!=k]) for k in range(K)])
        z     = (delta-delta.mean())/(delta.std()+1e-10)
        trust = np.exp(-np.maximum(z-1,0))

        # intra-cluster aggregation
        sz   = np.array(sizes,dtype=float)
        nrm  = np.linalg.norm(desc,axis=1,keepdims=True)
        phi  = desc/(nrm+1e-10)
        cflat= {}
        for cj in range(self.nc):
            mb = np.where(asgn==cj)[0]
            if len(mb)==0: continue
            cen = phi[mb].mean(0); cen /= (np.linalg.norm(cen)+1e-10)
            dd  = np.linalg.norm(phi[mb]-cen, axis=1)
            w   = sz[mb]*np.exp(-dd)*trust[mb]; w /= w.sum()+1e-10
            cflat[cj] = (w[:,None]*flat[mb]).sum(0)

        # inter-cluster blending
        ids  = list(cflat.keys())
        csz  = np.array([sz[asgn==cj].sum() for cj in ids])
        cw   = csz/(csz.sum()+1e-10)
        gf   = sum(w*cflat[cj] for w,cj in zip(cw,ids))
        b    = self.bc
        for cj in ids: cflat[cj] = (1-b)*cflat[cj]+b*gf

        csd = {cj: unflatten_params(v,ref) for cj,v in cflat.items()}
        return unflatten_params(gf,ref), csd, dict(mode='topo',asgn=asgn,trust=trust)

    def _cluster(self, desc):
        K=len(desc); n=min(self.nc,K)
        if n<2: return np.zeros(K,int)
        nrm=np.linalg.norm(desc,axis=1,keepdims=True)
        return AgglomerativeClustering(n_clusters=n,linkage='average').fit_predict(
            desc/(nrm+1e-10))

In [ ]:
from torchvision.models import resnet18
from torchvision import datasets, transforms

def get_resnet18(nc=10):
    m=resnet18(weights=None); m.fc=nn.Linear(512,nc); return m

class ConvNet2(nn.Module):
    def __init__(self, nc=62):
        super().__init__()
        self.conv=nn.Sequential(
            nn.Conv2d(1,32,5,padding=2),nn.ReLU(),nn.MaxPool2d(2),
            nn.Conv2d(32,64,5,padding=2),nn.ReLU(),nn.MaxPool2d(2))
        self.fc=nn.Sequential(
            nn.Linear(64*7*7,512),nn.ReLU(),nn.Dropout(0.3),nn.Linear(512,nc))
    def forward(self,x): return self.fc(self.conv(x).flatten(1))
    def penultimate(self,x): return self.fc[:-1](self.conv(x).flatten(1))

In [ ]:
@torch.no_grad()
def multiscale_resnet(model, images, n_sub=200):
    """Concat layer2(128)+layer3(256)+avgpool(512) = 896-dim activations."""
    model.eval().to(DEVICE)
    if len(images)>n_sub:
        images=images[np.random.choice(len(images),n_sub,replace=False)]
    x=torch.FloatTensor(images).to(DEVICE)
    x=model.relu(model.bn1(model.conv1(x))); x=model.maxpool(x)
    x=model.layer1(x)
    h2=model.layer2(x); h3=model.layer3(h2); h4=model.layer4(h3)
    hp=model.avgpool(h4).flatten(1)
    l2=h2.mean([2,3]); l3=h3.mean([2,3])
    return torch.cat([l2,l3,hp],1).cpu().numpy()

@torch.no_grad()
def multiscale_convnet(model, images, n_sub=200):
    model.eval().to(DEVICE)
    if len(images)>n_sub:
        images=images[np.random.choice(len(images),n_sub,replace=False)]
    x=torch.FloatTensor(images).to(DEVICE)
    h=model.conv(x).flatten(1); hp=model.fc[:-1](h)
    return torch.cat([h,hp],1).cpu().numpy()

In [ ]:
def train_basic(model, loader, n_ep, lr):
    model.train().to(DEVICE)
    opt=optim.SGD(model.parameters(),lr=lr,momentum=0.9,weight_decay=1e-4)
    crit=nn.CrossEntropyLoss()
    for _ in range(n_ep):
        for xb,yb in loader:
            xb,yb=xb.to(DEVICE),yb.to(DEVICE)
            opt.zero_grad(); crit(model(xb),yb).backward(); opt.step()
    return model.state_dict()

def train_fedprox(model, loader, global_sd, n_ep, lr, mu=0.1, model_fn_factory=None):
    model.train().to(DEVICE)
    opt=optim.SGD(model.parameters(),lr=lr,momentum=0.9,weight_decay=1e-4)
    crit=nn.CrossEntropyLoss()

    # Instantiate a temporary global model instance to extract only parameters
    # This ensures that `global_flat_params` (gf) corresponds only to trainable parameters.
    if model_fn_factory is not None:
        global_model_instance = model_fn_factory()
    else:
        # Fallback if model_fn_factory is not provided (e.g. for LR models)
        global_model_instance = type(model)()

    global_model_instance.load_state_dict(global_sd)
    gf_params = [p.flatten().float() for p in global_model_instance.parameters()]
    gf = torch.cat(gf_params).to(DEVICE)

    for _ in range(n_ep):
        for xb,yb in loader:
            xb,yb=xb.to(DEVICE),yb.to(DEVICE)
            opt.zero_grad()
            loc_params = [p.flatten() for p in model.parameters()]
            loc = torch.cat(loc_params) # This is already parameters only
            (crit(model(xb),yb)+(mu/2)*torch.sum((loc-gf)**2)).backward()
            opt.step()
    return model.state_dict()

def train_scaffold(model, loader, c_i, c_g, n_ep, lr):
    model.train().to(DEVICE)
    opt=optim.SGD(model.parameters(),lr=lr)
    crit=nn.CrossEntropyLoss()
    cd=torch.FloatTensor(c_g-c_i).to(DEVICE)
    w0=flatten_params(model.state_dict()).copy(); steps=0
    for _ in range(n_ep):
        for xb,yb in loader:
            xb,yb=xb.to(DEVICE),yb.to(DEVICE)
            opt.zero_grad(); crit(model(xb),yb).backward()
            off=0
            with torch.no_grad():
                for p in model.parameters():
                    n=p.numel()
                    if p.grad is not None:
                        p.grad.add_(cd[off:off+n].reshape(p.shape))
                    off+=n
            opt.step(); steps+=1
    w1=flatten_params(model.state_dict())
    new_ci=c_i-c_g+(w0-w1)/(steps*lr+1e-12)
    return model.state_dict(), new_ci

def train_pfedme(model, loader, global_sd, n_ep, lr, lam=15.0):
    model.train().to(DEVICE)
    opt=optim.SGD(model.parameters(),lr=lr)
    crit=nn.CrossEntropyLoss()
    gf=torch.FloatTensor(flatten_params(global_sd)).to(DEVICE)
    for _ in range(n_ep):
        for xb,yb in loader:
            xb,yb=xb.to(DEVICE),yb.to(DEVICE)
            opt.zero_grad()
            loc=torch.cat([p.flatten() for p in model.parameters()])
            (crit(model(xb),yb)+(lam/2)*torch.sum((loc-gf)**2)).backward()
            opt.step()
    return model.state_dict()

def fedavg_agg(sds, sizes):
    total=sum(sizes); w=np.array(sizes,dtype=float)/total
    flat=np.stack([flatten_params(sd) for sd in sds])
    return unflatten_params((w[:,None]*flat).sum(0), sds[0])

@torch.no_grad()
def eval_acc(model_fn, sd, loader):
    m=model_fn(); m.load_state_dict(sd); m.eval().to(DEVICE)
    correct=total=0
    for xb,yb in loader:
        xb,yb=xb.to(DEVICE),yb.to(DEVICE)
        correct+=(m(xb).argmax(1)==yb).sum().item(); total+=len(yb)
    return correct/(total+1e-9)


In [ ]:
def dirichlet_split(dataset, n_clients, alpha, seed=42):
    rng=np.random.default_rng(seed)
    targets=np.array(dataset.targets)
    client_idx=[[] for _ in range(n_clients)]
    for c in np.unique(targets):
        c_idx=np.where(targets==c)[0]; rng.shuffle(c_idx)
        props=rng.dirichlet(np.repeat(alpha,n_clients))
        props=(props*len(c_idx)).astype(int)
        props[-1]=len(c_idx)-props[:-1].sum()
        for k,s in enumerate(np.split(c_idx,np.cumsum(props)[:-1])):
            client_idx[k].extend(s.tolist())
    return client_idx

def cifar10_data(alpha=0.1, n_clients=10, data_dir='./data', bs=32):
    tf_tr=transforms.Compose([
        transforms.RandomCrop(32,padding=4),transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.4914,0.4822,0.4465),(0.2023,0.1994,0.2010))])
    tf_te=transforms.Compose([transforms.ToTensor(),
        transforms.Normalize((0.4914,0.4822,0.4465),(0.2023,0.1994,0.2010))])
    tr=datasets.CIFAR10(data_dir,True, download=True,transform=tf_tr)
    te=datasets.CIFAR10(data_dir,False,download=True,transform=tf_te)
    idx_list=dirichlet_split(tr,n_clients,alpha)
    loaders=[DataLoader(Subset(tr,idx),bs,shuffle=True,drop_last=True)
             for idx in idx_list]
    images=[np.stack([tr[i][0].numpy() for i in idx]) for idx in idx_list]
    return loaders,[len(i) for i in idx_list],images,DataLoader(te,256,shuffle=False)

def synthetic_femnist(n_clients=50, samples=300, nc=62, seed=42):
    rng=np.random.default_rng(seed)
    Xs,ys=[],[]
    for k in range(n_clients):
        cls=rng.choice(nc,size=rng.integers(3,10),replace=False)
        Xs.append(rng.standard_normal((samples,1,28,28)).astype(np.float32))
        ys.append(rng.choice(cls,size=samples).astype(np.int64))
    return Xs,ys

def femnist_data(n_clients=50, bs=32, leaf_dir=None):
    Xs,ys = synthetic_femnist(n_clients+5)  # last 5 = test
    tX=np.concatenate(Xs[-5:]); ty=np.concatenate(ys[-5:])
    Xs,ys=Xs[:n_clients],ys[:n_clients]
    loaders=[DataLoader(TensorDataset(torch.FloatTensor(X),torch.LongTensor(y)),
                        bs,shuffle=True,drop_last=True) for X,y in zip(Xs,ys)]
    test_ldr=DataLoader(TensorDataset(torch.FloatTensor(tX),torch.LongTensor(ty)),
                        256,shuffle=False)
    return loaders,[len(X) for X in Xs],Xs,test_ldr,n_clients

In [ ]:
METHODS = ['pTopoFL','FedAvg','FedProx','SCAFFOLD','pFedMe']
COLORS  = {'pTopoFL':'#0d9e7e','FedAvg':'#2563eb','FedProx':'#f59e0b',
            'SCAFFOLD':'#ef4444','pFedMe':'#8b5cf6'}
MARKERS = {'pTopoFL':'o','FedAvg':'s','FedProx':'^','SCAFFOLD':'D','pFedMe':'v'}

def run_fl_deep(tag, model_fn, nc, loaders, sizes, images, test_ldr,
                n_rounds, n_ep, lr, n_sub, is_resnet, warmup, seed=42):
    set_seed(seed)
    K=len(loaders)
    ext=TopologicalFeatureExtractor(n_points=n_sub,use_ripser=True)
    srv=TopoFLServer(n_clusters=3 if K>=10 else 2, blend_coeff=0.3,
                     anomaly_tau=2.0, warmup_rounds=warmup,
                     tda_kwargs=dict(n_points=n_sub))

    init_sd   = model_fn().state_dict()
    topo_sd   = copy.deepcopy(init_sd)
    base_sd   = {m: copy.deepcopy(init_sd) for m in
                 ['FedAvg','FedProx','SCAFFOLD','pFedMe']}
    np_      = sum(p.numel() for p in model_fn().parameters())
    cg_sc    = np.zeros(np_); cl_sc=[np.zeros(np_) for _ in range(K)]
    hist     = {m:[] for m in METHODS}
    act_fn   = multiscale_resnet if is_resnet else multiscale_convnet
    mfn      = lambda: model_fn()

    print(f"\n{'='*60}\n  {tag}  K={K}  rounds={n_rounds}  warmup={warmup}\n{'='*60}")
    t0=time.time()

    for rnd in tqdm(range(n_rounds), desc=tag):
        # ── pTopoFL ──
        tl_sds=[]
        for loader in loaders:
            m=mfn(); m.load_state_dict(topo_sd)
            tl_sds.append(train_basic(m,loader,n_ep,lr))
        bb=mfn(); bb.load_state_dict(topo_sd)
        descs=[ext.fit_transform(act_fn(bb,imgs,n_sub)) for imgs in images]
        topo_sd,_,_ = srv.aggregate(descs,tl_sds,sizes,round_idx=rnd)
        hist['pTopoFL'].append(eval_acc(mfn,topo_sd,test_ldr))

        # ── FedAvg ──
        fa_sds=[]
        for loader in loaders:
            m=mfn(); m.load_state_dict(base_sd['FedAvg'])
            fa_sds.append(train_basic(m,loader,n_ep,lr))
        base_sd['FedAvg']=fedavg_agg(fa_sds,sizes)
        hist['FedAvg'].append(eval_acc(mfn,base_sd['FedAvg'],test_ldr))

        # ── FedProx ──
        fp_sds=[]
        for loader in loaders:
            m=mfn(); m.load_state_dict(base_sd['FedProx'])
            fp_sds.append(train_fedprox(m,loader,base_sd['FedProx'],n_ep,lr, model_fn_factory=mfn))
        base_sd['FedProx']=fedavg_agg(fp_sds,sizes)
        hist['FedProx'].append(eval_acc(mfn,base_sd['FedProx'],test_ldr))

        # ── SCAFFOLD ──
        sc_sds,new_ci=[],[]
        for k,loader in enumerate(loaders):
            m=mfn(); m.load_state_dict(base_sd['SCAFFOLD'])
            sd,ci=train_scaffold(m,loader,cl_sc[k],cg_sc,n_ep,lr)
            sc_sds.append(sd); new_ci.append(ci)
        cg_sc=np.mean(new_ci,0); cl_sc=new_ci
        base_sd['SCAFFOLD']=fedavg_agg(sc_sds,sizes)
        hist['SCAFFOLD'].append(eval_acc(mfn,base_sd['SCAFFOLD'],test_ldr))

        # ── pFedMe ──
        pm_sds=[]
        for loader in loaders:
            m=mfn(); m.load_state_dict(base_sd['pFedMe'])
            pm_sds.append(train_pfedme(m,loader,base_sd['pFedMe'],n_ep,lr))
        base_sd['pFedMe']=fedavg_agg(pm_sds,sizes)
        hist['pFedMe'].append(eval_acc(mfn,base_sd['pFedMe'],test_ldr))

        if (rnd+1)%10==0 or rnd==0:
            tqdm.write(f"  Rnd {rnd+1:3d} ({(time.time()-t0)/60:.1f}min) "
                       + " ".join(f"{m[:4]}:{hist[m][-1]:.3f}" for m in METHODS))

    print(f"\n  {'Method':10s} {'Final':>7s} {'Best':>7s}")
    for m in METHODS:
        print(f"  {m:10s} {hist[m][-1]:>7.4f} {max(hist[m]):>7.4f}"
              + (' ◀' if m=='pTopoFL' else ''))
    return hist


In [ ]:
# ✖ Adjust N_ROUNDS_C: use 30 for a quick test, 100 for publication results
N_ROUNDS_C  = 100    # 100 for paper
N_EPOCHS_C  = 5
LR_C        = 0.01
WARMUP_C    = 10     # FIX 2: warm-up rounds

cifar_hist = {}
for ALPHA in [0.1, 0.5]:
    print(f"\n{'▶'*3} CIFAR-10  α={ALPHA}")
    loaders_c, sizes_c, images_c, test_c = cifar10_data(
        alpha=ALPHA, n_clients=10)
    cifar_hist[ALPHA] = run_fl_deep(
        tag       = f"CIFAR-10 α={ALPHA}",
        model_fn  = lambda: get_resnet18(10),
        nc        = 10,
        loaders   = loaders_c,
        sizes     = sizes_c,
        images    = images_c,
        test_ldr  = test_c,
        n_rounds  = N_ROUNDS_C,
        n_ep      = N_EPOCHS_C,
        lr        = LR_C,
        n_sub     = 200,
        is_resnet = True,
        warmup    = WARMUP_C)


▶▶▶ CIFAR-10  α=0.1


100%|██████████| 170M/170M [00:14<00:00, 11.9MB/s]



  CIFAR-10 α=0.1  K=10  rounds=100  warmup=10


CIFAR-10 α=0.1:   0%|          | 0/100 [00:00<?, ?it/s]

  Rnd   1 (21.8min) pTop:0.100 FedA:0.162 FedP:0.142 SCAF:0.180 pFed:0.137


In [ ]:
N_ROUNDS_D = 200     # 200 for paper
N_EPOCHS_D = 3
LR_D       = 0.01
WARMUP_D   = 15

print(f"\n{'▶'*3} FEMNIST  50 clients")
loaders_d, sizes_d, images_d, test_d, K_d = femnist_data(n_clients=50)
femnist_hist = run_fl_deep(
    tag       = "FEMNIST 50-client",
    model_fn  = lambda: ConvNet2(62),
    nc        = 62,
    loaders   = loaders_d,
    sizes     = sizes_d,
    images    = images_d,
    test_ldr  = test_d,
    n_rounds  = N_ROUNDS_D,
    n_ep      = N_EPOCHS_D,
    lr        = LR_D,
    n_sub     = 200,
    is_resnet = False,
    warmup    = WARMUP_D)

In [ ]:
print("\n" + "="*70)
print("TABLE 2 — Deep model results (100 / 200 rounds)")
print(f"{'Method':12s} | {'CIFAR-10 α=0.1':>14s} | {'CIFAR-10 α=0.5':>14s} "
      f"| {'FEMNIST':>10s}")
print("-"*58)
for m in METHODS:
    c1  = cifar_hist[0.1][m][-1]
    c5  = cifar_hist[0.5][m][-1]
    fe  = femnist_hist[m][-1]
    mrk = ' ◀' if m=='pTopoFL' else ''
    print(f"{m:12s} | {c1:>14.4f} | {c5:>14.4f} | {fe:>10.4f}{mrk}")
print("="*70)

In [ ]:
fig = plt.figure(figsize=(18, 5))
gs  = gridspec.GridSpec(1, 3, figure=fig, wspace=0.35)
titles = [f'Scenario C — CIFAR-10 (α=0.1)',
          f'Scenario C — CIFAR-10 (α=0.5)',
          'Scenario D — FEMNIST']
datasets_plot = [cifar_hist[0.1], cifar_hist[0.5], femnist_hist]

for ax_idx, (hist, title) in enumerate(zip(datasets_plot, titles)):
    ax = fig.add_subplot(gs[ax_idx])
    for m in METHODS:
        vals = hist[m]
        rounds = range(1, len(vals)+1)
        ax.plot(rounds, vals, color=COLORS[m], marker=MARKERS[m],
                markevery=max(1,len(vals)//10), markersize=5,
                linewidth=2.0, label=m)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel('FL Round', fontsize=11)
    ax.set_ylabel('Top-1 Accuracy', fontsize=11)
    ax.legend(fontsize=8, loc='lower right')
    ax.grid(alpha=0.3)
    # shade warm-up region
    wu = WARMUP_C if 'FEMNIST' not in title else WARMUP_D
    ax.axvspan(1, wu, alpha=0.07, color='grey', label='warm-up')

fig.suptitle('pTopoFL vs Baselines — Deep Model Experiments',
             fontsize=13, fontweight='bold', y=1.02)
plt.savefig('fig_deep_experiments.pdf', bbox_inches='tight', dpi=150)
plt.show()
print("✓  Saved fig_deep_experiments.pdf")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (hist, tag, wu) in zip(axes, [
        (cifar_hist[0.1], 'CIFAR-10 α=0.1', WARMUP_C),
        (cifar_hist[0.5], 'CIFAR-10 α=0.5', WARMUP_C),
        (femnist_hist,    'FEMNIST',          WARMUP_D)]):
    topo = np.array(hist['pTopoFL'])
    best_base = np.array([max(hist[m][r] for m in METHODS if m!='pTopoFL')
                          for r in range(len(topo))])
    advantage = topo - best_base
    rounds = range(1, len(topo)+1)
    ax.plot(rounds, advantage, color='#0d9e7e', linewidth=2)
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax.axvspan(1, wu, alpha=0.07, color='grey')
    ax.fill_between(rounds, advantage, 0,
                    where=advantage>0, alpha=0.2, color='#0d9e7e',
                    label='pTopoFL ahead')
    ax.fill_between(rounds, advantage, 0,
                    where=advantage<0, alpha=0.2, color='#ef4444',
                    label='pTopoFL behind')
    crossover = next((r+1 for r,v in enumerate(advantage) if v>0), None)
    if crossover:
        ax.axvline(crossover, color='#0d9e7e', linestyle=':', linewidth=1.5,
                   label=f'crossover rnd {crossover}')
    ax.set_title(f'{tag}\nAdvantage over best baseline', fontsize=10,
                 fontweight='bold')
    ax.set_xlabel('FL Round'); ax.set_ylabel('Accuracy gap')
    ax.legend(fontsize=7); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('fig_advantage.pdf', bbox_inches='tight', dpi=150)
plt.show()
print("✓  Saved fig_advantage.pdf")

In [ ]:
os.makedirs('./results', exist_ok=True)
all_results = {
    'cifar10_alpha01': cifar_hist[0.1],
    'cifar10_alpha05': cifar_hist[0.5],
    'femnist':         femnist_hist,
    'table2': {
        m: {
            'cifar10_alpha01': cifar_hist[0.1][m][-1],
            'cifar10_alpha05': cifar_hist[0.5][m][-1],
            'femnist':         femnist_hist[m][-1],
        } for m in METHODS
    }
}
with open('./results/deep_results.json', 'w') as f:
    json.dump(all_results, f, indent=2)
print("✓  All results saved to ./results/deep_results.json")
print("\nCopy the table2 section directly into your LaTeX.")